# Inspect the local Qdrant collection


In [15]:
import pandas as pd
from qdrant_client import QdrantClient, models
from ingestion import DB_PATH_NAME
import sys
from pathlib import Path

In [16]:
# Set the Collection we want to inspect
COLLECTION_NAME = 'sec_filings_openai_text-embedding-3-small_v1_2'
DB_PATH = DB_PATH_NAME
client = QdrantClient(path=str(DB_PATH))


## Inspection functions

In [17]:
def list_collections(qdrant_client: QdrantClient) -> list[str]:
    return [
        collection.name
        for collection in qdrant_client.get_collections().collections
    ]


def collection_summary(
    qdrant_client: QdrantClient,
    collection_name: str,
) -> dict:
    if not qdrant_client.collection_exists(collection_name):
        raise ValueError(f"Collection does not exist: {collection_name}")

    info = qdrant_client.get_collection(collection_name)
    vector_config = info.config.params.vectors
    exact_count = qdrant_client.count(
        collection_name=collection_name,
        exact=True,
    ).count

    return {
        "collection_name": collection_name,
        "point_count": exact_count,
        "vector_size": vector_config.size,
        "distance": str(vector_config.distance),
        "status": str(info.status),
    }


def sample_points(
    qdrant_client: QdrantClient,
    collection_name: str,
    limit: int = 5,
) -> pd.DataFrame:
    records, _ = qdrant_client.scroll(
        collection_name=collection_name,
        limit=limit,
        with_payload=True,
        with_vectors=False,
    )

    rows = []
    for record in records:
        payload = record.payload or {}
        text = payload.get("chunk_text", "")
        rows.append({
            "qdrant_id": str(record.id),
            "chunk_id": payload.get("chunk_id"),
            "ticker": payload.get("ticker"),
            "form_type": payload.get("form_type"),
            "filing_date": payload.get("filing_date"),
            "section": payload.get("chunk_title"),
            "chunk_type": payload.get("chunk_type"),
            "table_id": payload.get("table_id"),
            "text_preview": text[:300],
        })

    return pd.DataFrame(rows)


def count_matching(
    qdrant_client: QdrantClient,
    collection_name: str,
    field: str,
    value: str,
) -> int:
    query_filter = models.Filter(
        must=[
            models.FieldCondition(
                key=field,
                match=models.MatchValue(value=value),
            )
        ]
    )

    return qdrant_client.count(
        collection_name=collection_name,
        count_filter=query_filter,
        exact=True,
    ).count


def counts_by_values(
    qdrant_client: QdrantClient,
    collection_name: str,
    field: str,
    values: list[str],
) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                field: value,
                "count": count_matching(
                    qdrant_client, collection_name, field, value
                ),
            }
            for value in values
        ]
    )


def inspect_one_vector(
    qdrant_client: QdrantClient,
    collection_name: str,
) -> dict:
    records, _ = qdrant_client.scroll(
        collection_name=collection_name,
        limit=1,
        with_payload=False,
        with_vectors=True,
    )

    if not records:
        return {"has_vector": False}

    vector = records[0].vector
    return {
        "has_vector": vector is not None,
        "dimensions": len(vector) if vector is not None else None,
        "first_five_values": vector[:5] if vector is not None else None,
    }


def integrity_report(
    qdrant_client: QdrantClient,
    collection_name: str,
) -> dict:
    summary = collection_summary(qdrant_client, collection_name)
    vector = inspect_one_vector(qdrant_client, collection_name)
    text_count = count_matching(
        qdrant_client, collection_name, "chunk_type", "text"
    )
    table_count = count_matching(
        qdrant_client, collection_name, "chunk_type", "table"
    )

    return {
        **summary,
        "text_chunks": text_count,
        "table_chunks": table_count,
        "known_chunk_types_total": text_count + table_count,
        "chunk_type_counts_match_total": (
            text_count + table_count == summary["point_count"]
        ),
        "sample_vector_dimensions": vector.get("dimensions"),
        "vector_size_matches_collection": (
            vector.get("dimensions") == summary["vector_size"]
        ),
    }

## Collection overview

In [18]:
pd.Series(collection_summary(client, COLLECTION_NAME), name="value")

collection_name    sec_filings_openai_text-embedding-3-small_v1_2
point_count                                                  8562
vector_size                                                  1536
distance                                                   Cosine
status                                                      green
Name: value, dtype: object

## Inspect sample payloads

In [19]:
sample_points(client, COLLECTION_NAME, limit=5)

,qdrant_id,chunk_id,ticker,form_type,filing_date,section,chunk_type,table_id,text_preview
0,0006a3bb-7110-557e-9521-d8761fb0f040,0001193125-24-130897*section-2*text-0003,ETHE,10-Q,2024-05-03,Management’s Discussion and Analysis of Financ...,text,None,Section: Management’s Discussion and Analysis ...
1,00124aff-adaf-57ff-91b9-6cd26398560a,0001437749-24-006339*section-1*text-0043,IBIT,10-K,2024-03-04,Business,text,None,Section: Business\n\nThe Basket Amount necessa...
2,0012997d-45ac-5532-9fce-b2fb9cf2630d,0001193125-22-053501*section-4*text-0000,GBTC,10-K,2022-02-25,Mine Safety Disclosures,text,None,Section: Mine Safety Disclosures\n\nNot applic...
3,0018dcc3-9cc4-5f36-86c7-c6806b14fe56,0000950170-24-120130*section-6*text-0000,ETHE,10-Q,2024-11-01,Exhibits | 30,text,None,Section: Exhibits | 30\n\nGLOSSARY OF DEFINED ...
4,001c3450-9b0f-5153-926f-e6956492c40d,0001437749-26-006058*section-1*text-0071,IBIT,10-K,2026-02-27,Business,text,None,Section: Business\n\nBrokerage Fees and Trust ...


## Count chunks by type and ticker

In [20]:
counts_by_values(
    client,
    COLLECTION_NAME,
    field="chunk_type",
    values=["text", "table"],
)

,chunk_type,count
0,text,5938
1,table,2624


In [21]:
counts_by_values(
    client,
    COLLECTION_NAME,
    field="ticker",
    values=["IBIT", "ETHA", "FBTC", "FETH", "GBTC", "ETHE"],
)

,ticker,count
0,IBIT,1599
1,ETHA,1336
2,FBTC,626
3,FETH,584
4,GBTC,3037
5,ETHE,1380


## Check one stored vector

This returns only the vector dimensions and its first five values.

In [22]:
inspect_one_vector(client, COLLECTION_NAME)

{'has_vector': True,
 'dimensions': 1536,
 'first_five_values': [0.03948974609375,
  0.0164642333984375,
  0.0207672119140625,
  0.0158843994140625,
  -0.01172637939453125]}

## Basic integrity report

In [23]:
pd.Series(integrity_report(client, COLLECTION_NAME), name="value")

collection_name                   sec_filings_openai_text-embedding-3-small_v1_2
point_count                                                                 8562
vector_size                                                                 1536
distance                                                                  Cosine
status                                                                     green
text_chunks                                                                 5938
table_chunks                                                                2624
known_chunk_types_total                                                     8562
chunk_type_counts_match_total                                               True
sample_vector_dimensions                                                    1536
vector_size_matches_collection                                              True
Name: value, dtype: object

## Close the local database

Run this cell when you finish. Embedded Qdrant allows only one process to hold the storage lock.

In [24]:
client.close()
print("Qdrant client closed.")

Qdrant client closed.
